Qué es train_test_split:

Es simplemente dividir los datos en dos partes:

1️⃣ Train (entrenamiento)
Datos que el modelo usa para aprender.

2️⃣ Test (prueba)
Datos que el modelo nunca vio y que usamos para ver si realmente aprendió.

In [66]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, roc_auc_score


In [67]:
df = pd.read_csv("../data/telco_churn_clean.csv")

In [68]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [69]:
y.shape

(7043,)

In [70]:
X.shape

(7043, 30)

Reparto entrenamiento y prueba

In [71]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
    #80% → entrenamiento
    #20% → prueba
)

Apliacion del Scaling

In [72]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Entrenamiento del modelo


In [73]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [74]:
y_pred = model.predict(X_test_scaled)
y_pred

array([1, 0, 0, ..., 0, 0, 0], shape=(1409,))

In [75]:
accuracy_score(y_test, y_pred)

0.8204400283889283

In [76]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.69      0.60      0.64       373

    accuracy                           0.82      1409
   macro avg       0.77      0.75      0.76      1409
weighted avg       0.81      0.82      0.82      1409



or filas (por clase)
Cada fila es una clase:

Clase 0 → clientes que no hacen churn.
Clase 1 → clientes que sí hacen churn.
Para la clase 0
precision 0.86
De todos los clientes que el modelo dijo “no se van” (predicción 0), el 86 % realmente no se va.

recall 0.90
De todos los clientes que realmente no se van, el modelo encontró correctamente al 90 % (solo se equivocó con el 10 %).

f1-score 0.88
Es un promedio entre precision y recall → 0.88 indica un muy buen rendimiento en la clase 0.

support 1036
En el conjunto de test hay 1036 clientes reales de clase 0.

Para la clase 1 (la importante: churn)
precision 0.69
De todos los que el modelo dijo “este cliente se va” (predicción 1), el 69 % realmente se va.
El 31 % restante son falsos positivos (el modelo pensó que se iban y no).

recall 0.60
De todos los clientes que realmente se van, el modelo detecta solo el 60 %.
El otro 40 % son falsos negativos (se iban, pero el modelo dijo que no).

f1-score 0.64
Resumen entre precisión y recall. 0.64 es aceptable pero claramente más bajo que la clase 0.

support 373
En el test hay 373 clientes reales con churn = 1.

In [77]:
confusion_matrix(y_test, y_pred)
#[[TN FP]
# [FN TP]]

array([[934, 102],
       [151, 222]])

934 (TN)
934 clientes no iban a hacer churn y el modelo dijo “no churn” → aciertos tranquilos.

102 (FP)
102 clientes no iban a hacer churn, pero el modelo dijo “sí churn”.
Son clientes que ibas a marcar como “en riesgo” sin que realmente lo estén (alarmas falsas).

151 (FN)
151 clientes sí iban a hacer churn, pero el modelo dijo “no churn”.
Estos son los peligrosos para el negocio: clientes que se van y el modelo no los detecta → no los retienes.

222 (TP)
222 clientes sí iban a hacer churn y el modelo dijo “sí churn”.
Son los clientes en riesgo que el modelo detecta bien y a los que podrías intentar retener.

En resumen:

El modelo acierta mucho con los que se quedan (934).
Detecta parte de los que se van (222), pero se le escapan 151 que también se iban.
Si para la empresa es peor perder clientes que molestar a alguno de más, querrás reducir esos 151 FN, aunque aumenten algo los 102 FP.

Vamos a calcular ROC-AUC, que es una métrica clave en clasificación. -------------->>
Esto nos dirá qué tan bien el modelo separa churn vs no churn.

In [78]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]
roc_auc = roc_auc_score(y_test, y_prob)
roc_auc

0.8621256741229931

Analiza que variables causan Churn importante para las empresas.

Feature Importance / Coeficientes del modelo

In [79]:
coef = pd.DataFrame({
    "feature": X.columns,
    "coef": model.coef_[0]
})

coef["abs_coef"] = coef["coef"].abs()

coef.sort_values("abs_coef", ascending=False).head(10)

,feature,coef,abs_coef
1,tenure,-1.347355,1.347355
3,TotalCharges,0.649427,0.649427
2,MonthlyCharges,-0.628014,0.628014
10,InternetService_Fiber optic,0.619853,0.619853
25,Contract_Two year,-0.613702,0.613702
24,Contract_One year,-0.268336,0.268336
23,StreamingMovies_Yes,0.228714,0.228714
21,StreamingTV_Yes,0.179380,0.179380
9,MultipleLines_Yes,0.168199,0.168199
26,PaperlessBilling_Yes,0.163829,0.163829


In [80]:
fn = X_test[(y_test == 1) & (y_pred == 0)]
fn.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
1263,1,68,89.60,6127.60,False,True,False,True,False,True,...,False,False,False,True,False,False,True,False,False,False
811,0,70,104.00,7250.15,True,False,False,True,False,True,...,False,True,False,True,False,True,True,True,False,False
2526,0,1,19.40,19.40,True,False,False,True,False,False,...,True,False,True,False,False,False,False,False,False,True
5275,0,11,53.75,608.00,True,False,False,True,False,True,...,False,False,False,False,False,False,False,False,True,False
5194,0,22,89.40,2001.50,True,False,True,True,False,False,...,False,True,False,False,False,False,True,False,True,False


In [95]:
y_prob = model.predict_proba(X_test_scaled)[:, 1]

threshold = 0.35  # prueba 0.5, 0.4, 0.35, 0.3...
y_pred_new = (y_prob >= threshold).astype(int)

print(classification_report(y_test, y_pred_new))
print(confusion_matrix(y_test, y_pred_new))

              precision    recall  f1-score   support

           0       0.90      0.80      0.85      1036
           1       0.57      0.74      0.64       373

    accuracy                           0.78      1409
   macro avg       0.73      0.77      0.74      1409
weighted avg       0.81      0.78      0.79      1409

[[829 207]
 [ 97 276]]


Ajustando el umbral de decisión, el modelo mantiene una accuracy cercana al 80 %, pero aumenta el recall de la clase churn del 60 % al 74 %.
Esto significa que detectamos muchos más clientes que van a cancelar, a costa de marcar como ‘en riesgo’ a algunos que realmente se quedarían (falsos positivos), lo cual es asumible comparado con el coste de perder clientes que se van sin ser detectados.